##  第一阶段：启动与配置（Shell & Hydra）
- 负责拉起分布式集群，分配GPU资源

<b>Ray 负责分布式调度，Hydra 管理复杂配置，FSDP/Megatron 处理模型并行，模块化设计支持灵活扩展。</b>

In [ ]:
# 查看集群上所有作业
ray job list --address="http://..."

# 查看特定作业日志
ray job logs <job_id> --address="http://..."

# 停止作业
ray job stop <job_id> --address="http://..."

# 提交并跟踪日志（-f 表示 follow）
ray job submit --address="..." --follow -- python3 -m recipe.dapo.main_dapo

### 场景：提交 DAPO 训练作业到 Ray 集群

In [ ]:
# 完整命令
ray job submit \
    --address="http://192.168.1.100:8265" \
    --working-dir="/home/user/dapo_project" \
    --runtime-env-json='{
        "pip": ["hydra-core", "omegaconf", "torch", "ray[train]"],
        "env_vars": {"CUDA_VISIBLE_DEVICES": "0,1,2,3"}
    }' \
    -- \
    python3 -m recipe.dapo.main_dapo \
        training.batch_size=256 \
        optimizer.lr=0.0001 \
        model.name=llama2-7b \
        --multirun  # Hydra 批量实验

### python 入口加载 （recipe/dapo/main_dapo.py）

1. python解释器加载 main_dapo.py
2. 使用@hydra.main()装饰器读取config/dapo_trainer.yaml中的配置
3. 覆盖配置：将shell脚本传入的命令行参数合并到配置对象cfg中
4. 调用：<font color='red'>执行run_ppo(cfg)函数</font>
   - 初始化ray：ray.init()
   - TaskRunner，首先调用@ray.remote(num_cpus=1),将TaskRunner类转换为ray actor，并申请一个CPU，用于调度（head节点负责集群管理和调度，不运行计算任务，真正的计算在worker中进行）

#### OmegaConf.resolve(config)作用

In [ ]:
# 配置文件中可能有变量引用
paths:
  base: /data/models
  checkpoint: ${paths.base}/llama2-7b  # 未解析时是字符串

# resolve=True 后变成：
paths:
  base: /data/models
  checkpoint: /data/models/llama2-7b   # 实际值

#### 模型下载与 Tokenizer 初始化

copy_to_local 的作用：
- 集群节点可能无法直接访问 HDFS/S3
- 自动将模型权重下载到本地 SSD，加速后续读取

In [ ]:
# 从 HDFS/远程存储下载到本地
local_path = copy_to_local(config.actor_rollout_ref.model.path)

# 初始化 HuggingFace Tokenizer 和 Processor
from verl.utils import hf_processor, hf_tokenizer

tokenizer = hf_tokenizer(local_path)
processor = hf_processor(local_path, use_fast=True)  # 多模态 LLM 使用

#### 分布式策略选择（FSDP vs Megatron）

| 策略             | 适用场景          | 特点                |
| -------------- | ------------- | ----------------- |
| **FSDP/FSDP2** | 中小规模模型（< 70B） | PyTorch 原生，易于调试   |
| **Megatron**   | 大规模模型（> 70B）  | NVIDIA 优化，张量并行效率高 |


In [ ]:
from verl.single_controller.ray import RayWorkerGroup

if config.actor_rollout_ref.actor.strategy in {"fsdp", "fsdp2"}:
    assert config.critic.strategy in {"fsdp", "fsdp2"}
    # 加载fsdp组件
    from verl.workers.fsdp_workers import ActorRolloutRefWorker, CriticWorker
    ray_worker_group_cls = RayWorkerGroup

elif config.actor_rollout_ref.actor.strategy == "megatron":
    assert config.actor_rollout_ref.actor.strategy == config.critic.strategy
    # 加载megatron组件
    from verl.workers.megatron_workers import ActorRolloutRefWorker, CriticWorker
    ray_worker_group_cls = RayWorkerGroup

#### 5. 角色与资源映射（核心设计）

In [ ]:
from verl.trainer.ppo.ray_trainer import ResourcePoolManager, Role

# 定义角色到 Worker 类的映射
role_worker_mapping = {
    Role.ActorRollout: ray.remote(ActorRolloutRefWorker),  # 策略网络 + 采样
    Role.Critic: ray.remote(CriticWorker),                  # 价值网络
}

# 资源池规格：每个节点 n_gpus_per_node 个 GPU，共 nnodes 个节点
global_pool_id = "global_pool"
resource_pool_spec = {
    global_pool_id: [config.trainer.n_gpus_per_node] * config.trainer.nnodes,
    # 例如 2 节点 × 8 GPU = [8, 8]
}

# 角色分配到哪个资源池
mapping = {
    Role.ActorRollout: global_pool_id,
    Role.Critic: global_pool_id,
}

In [ ]:
# 资源池可视化
Resource Pool "global_pool": [8, 8]  # 2 节点，每节点 8 GPU

Role.ActorRollout  ──┐
Role.Critic        ──┼──► 共享这 16 个 GPU
Role.RewardModel   ──┘   (通过 Ray 调度动态分配)

### 可选组件动态加载
#### Reward Model（可选）

In [ ]:
if config.reward_model.enable:
    if config.reward_model.strategy in {"fsdp", "fsdp2"}:
        from verl.workers.fsdp_workers import RewardModelWorker # fsdp_workers
    elif config.reward_model.strategy == "megatron":
        from verl.workers.megatron_workers import RewardModelWorker # megatron_workers
    
    # 动态添加到角色映射
    role_worker_mapping[Role.RewardModel] = ray.remote(RewardModelWorker)
    mapping[Role.RewardModel] = global_pool_id

#### Reference Policy（用于 KL 散度）

条件加载逻辑：
- 只有启用 KL 惩罚时才需要 Reference Policy
- 避免不必要的内存和计算开销

In [ ]:
if config.algorithm.use_kl_in_reward or config.actor_rollout_ref.actor.use_kl_loss:
    # 复用 ActorRolloutRefWorker 作为参考策略
    role_worker_mapping[Role.RefPolicy] = ray.remote(ActorRolloutRefWorker)
    mapping[Role.RefPolicy] = global_pool_id

### 奖励函数管理器（多源奖励）

| 类型              | 用途 | 组成                         |
| --------------- | -- | -------------------------- |
| `reward_type=0` | 训练 | 模型奖励（RM 打分）+ 规则奖励（格式/长度检查） |
| `reward_type=1` | 验证 | 纯规则奖励（更稳定，避免 RM 过拟合）       |


In [ ]:
reward_fn = load_reward_manager(
    config,
    tokenizer,
    reward_type=0,  # 训练用：模型奖励 + 规则奖励组合
    max_resp_len=config.data.max_response_length,
    overlong_buffer_cfg=config.reward_model.overlong_buffer,
)

# 验证专用（纯规则-based，更稳定）
val_reward_fn = load_reward_manager(
    config,
    tokenizer,
    reward_type=1,  # 验证用：仅规则奖励
    max_resp_len=config.data.max_response_length,
    overlong_buffer_cfg=config.reward_model.overlong_buffer,
)

###  资源池管理与训练器初始化

In [1]:
resource_pool_manager = ResourcePoolManager(
    resource_pool_spec=resource_pool_spec,  # [8, 8]
    mapping=mapping                           # 角色→池映射
)

trainer = RayDAPOTrainer(
    config=config,
    tokenizer=tokenizer,
    processor=processor,
    role_worker_mapping=role_worker_mapping,        # 角色类定义
    resource_pool_manager=resource_pool_manager,    # GPU 资源分配
    ray_worker_group_cls=ray_worker_group_cls,      # RayWorkerGroup
    reward_fn=reward_fn,                             # 训练奖励
    val_reward_fn=val_reward_fn,                     # 验证奖励
)

NameError: name 'ResourcePoolManager' is not defined

### 启动训练

###### init_workers() 内部动作：
1. <font color='red'>根据 resource_pool_spec 在 Ray 集群上申请 GPU 资源</font>
2. 在每个 GPU 上实例化对应的 Worker Actor
3. <font color='red'>建立 Worker 之间的通信组（NCCL 集合通信）</font>
4. 加载模型权重到各 Worker

In [ ]:
trainer.init_workers()  # 初始化所有 Ray Worker（实际创建 Actor）
trainer.fit()           # 开始 DAPO 训练循环

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    DAPO Training Loop                       │
├─────────────────────────────────────────────────────────────┤
│  1. ActorRollout.generate()                                   │
│     └── 生成 responses（rollout）                           │
│         └── 使用当前策略 π_θ 采样动作                        │
│                                                               │
│  2. RewardModel.score() / RuleReward.compute()               │
│     └── 计算奖励 r_t = r_model + r_rule - β·KL(π_θ‖π_ref)   │
│                                                               │
│  3. Critic.estimate_value()                                   │
│     └── 计算状态价值 V(s_t)                                  │
│                                                               │
│  4. 计算优势函数 A(s_t, a_t) = GAE(returns, values)          │
│                                                               │
│  5. Actor.update_policy()                                     │
│     └── 最大化 PPO-clip 目标：L^CLIP(θ)                       │
│                                                               │
│  6. Critic.update_value()                                     │
│     └── 最小化价值损失：MSE(V(s_t), returns)                 │
│                                                               │
│  7. 重复直到收敛                                              │
└─────────────────────────────────────────────────────────────┘

# 第二阶段： 分布式环境初始化（ray & manager）

## Ray集群初始化（在run_ppo函数内）

- 动作：调用ray.init() (通常在trainer内部或之前）
- <font color='red'>实例化Trainer：创建RayDAPOTrainer对象</font>
    - 代码位置：verl/trainer/ppo/ray_trainer.py
- 资源池管理：
    - <font color='red'>调用ResourcePoolManager</font>
        - create_resource_pool 创建资源池
        - get_resource_pool
        - get_n_gpus
        - _check_resource_available
    - 根据配置trainer.n_gpus_per_node=8,NNODES=16计算总资源
    - 角色分配：将GPU划分为不同的角色组：actor,critic,reward,reference
    

###### ResourcePoolManager 是核心组件，负责管理 Ray 集群中的计算资源。

这三个参数共同定义了“有哪些资源”、“资源在哪里”以及“谁使用哪些资源”。以下是详细解析：

1. resource_pool_spec (资源池规格定义)

含义： 这是资源分配的蓝图或配置字典。它定义了系统中存在哪些<b>逻辑资源池</b>，以及每个池中包含多少物理计算资源（通常是 GPU 数量或进程数）。

- 作用： 告诉管理器需要创建哪些池子，以及每个池子的大小。
- 数据结构： 通常是一个字典 Dict[str, List[int]]。
    - Key (字符串): 资源池的名称（例如 'actor_pool', 'critic_pool'）。
    - Value (列表): 描述资源在不同节点上的分布。列表中的每个数字代表一个节点上分配的 GPU 数量或进程数。

In [ ]:
# 定义了两个池：actor_pool 和 critic_pool
# actor_pool: 分布在2个节点上，每个节点8个GPU (共16个GPU)
# critic_pool: 分布在1个节点上，有4个GPU
resource_pool_spec = {
    "actor_pool": [8, 8], 
    "critic_pool": [4]
}

2. mapping (角色 与 资源池的映射)

含义： 这是一个路由规则字典。<font color='red'>它将训练任务中的不同角色（Role，如 Actor, Critic, RewardModel）绑定到具体的资源池名称上。</font>

- 作用： 决定哪个计算任务去哪个池子里运行。通过修改 mapping，你可以灵活地让不同的模型共享资源，或者独占资源。
- 数据结构： 字典 Dict[Role, str]。
    - Key: 角色枚举或字符串（如 Role.ActorRollout）。
    - Value: 对应 resource_pool_spec 中定义的池子名称。
- 示例：

In [ ]:
mapping = {
    Role.ActorRollout: "actor_pool",  # Actor 任务去 actor_pool 运行
    Role.Critic: "critic_pool",       # Critic 任务去 critic_pool 运行
    Role.RefPolicy: "actor_pool"      # 参考策略也去 actor_pool (共享资源)
}

3. resource_pool_dict (资源池实例字典)

含义： 这是运行时状态容器。在 ResourcePoolManager 初始化并执行创建逻辑后，实际生成的 RayResourcePool 对象会被存储在这里。

- 作用： 它是程序运行时的“句柄”集合。<font color='red'>代码后续需要通过这个字典获取具体的资源池对象，进而创建 WorkerGroup 或提交任务。</font>
- 数据结构： 字典 Dict[str, RayResourcePool]。
    - Key: 资源池名称。
    - Value: <font color='red'>实例化后的 RayResourcePool 对象（包含 Ray 的 Actor 句柄、进程组信息等）。</font>
- 生命周期： 通常在代码运行初期是空的，调用 create_resource_pool() 方法后被填充。

| 参数名 | 核心含义 | 数据性质 | 典型内容示例 |
| :--- | :--- | :--- | :--- |
| resource_pool_spec | 规划图 | 配置/输入 | `{"pool_A": [8, 8], "pool_B": [4]}` |
| mapping | 分配规则 | 逻辑映射 | `{Actor: "pool_A", Critic: "pool_B"}` |
| resource_pool_dict | 实物仓库 | 运行时对象 | `{"pool_A": <RayResourcePool object at 0x...>, ...}` |


<font color='red'>训练器根据 mapping 找到对应的池子名称，去 resource_pool_dict 取出对象，并启动具体的 Worker。</font>

In [ ]:
def create_resource_pool(self): # 创建资源池
        """Create Ray resource pools for distributed training.

        Initializes resource pools based on the resource pool specification,
        with each pool managing GPU resources across multiple nodes.
        For FSDP backend, uses max_colocate_count=1 to merge WorkerGroups.
        For Megatron backend, uses max_colocate_count>1 for different models.
        """
        for resource_pool_name, process_on_nodes in self.resource_pool_spec.items():
            # max_colocate_count means the number of WorkerGroups (i.e. processes) in each RayResourcePool
            # For FSDP backend, we recommend using max_colocate_count=1 that merge all WorkerGroups into one.
            # For Megatron backend, we recommend using max_colocate_count>1
            # that can utilize different WorkerGroup for differnt models
            # 定义每个actor的资源池
            resource_pool = RayResourcePool(
                process_on_nodes=process_on_nodes,
                use_gpu=True,
                max_colocate_count=1,
                name_prefix=resource_pool_name,
            )
            self.resource_pool_dict[resource_pool_name] = resource_pool

        self._check_resource_available()

### ray.init()

ray.init() 是 Ray 框架的入口点和总开关。你可以把它想象成电脑的“电源键”——只有按下了它，Ray 的分布式运行时（Runtime）才会启动，<b>你的代码才能从单机模式切换到分布式模式。</b>

这个看似简单的函数调用，实际上在后台完成了一系列复杂的资源编排和环境初始化工作。

#### 以下是 ray.init() 的详细技术拆解：

1. <font color='red'>核心功能：它在后台做了什么？</font>

当你调用 ray.init() 时，它主要执行了以下几个关键步骤：

 - <font color='red'>启动 Ray 集群组件：</font>
    - 它会启动 GCS（全局控制服务，基于 Redis），用于存储集群元数据和状态。
    - <b>启动 Raylet（节点代理），负责管理本节点的资源（CPU/GPU）和任务调度。</b>
    - 启动 Object Store（Plasma 对象存储），用于在进程间高效共享大块数据。
 - <font color='red'>资源注册与锁定：</font>
    - 它会检测当前机器的硬件资源（CPU 核心数、GPU 数量、内存大小），<b>并将这些资源注册到 GCS 中，供调度器分配。</b>
 - <font color='red'>建立连接上下文：</font>
    - <b>初始化 Python 客户端与 Ray 后端的连接，使得后续的 @ray.remote 装饰器和 ray.get() 等操作能够正常工作。</b>
    

2. 三种启动模式（场景）

ray.init() 非常智能，根据传入参数的不同，它会表现出三种完全不同的行为模式：
| 模式 | 代码示例 | 行为描述 | 适用场景 |
| :--- | :--- | :--- | :--- |
| 本地单机模式 | `ray.init()` | 自动创建集群。在当前机器上启动一个全新的 Ray 集群（Head Node + Worker Node）。 | 本地开发、调试、单元测试。 |
| 连接现有集群 | `ray.init(address='auto')` | <b>加入集群</b>。自动查找并连接到本机已存在的 Ray 集群（通常是通过命令行 `ray start` 启动的）。 | 在生产环境中，脚本作为 Worker 提交任务。 |
| 远程客户端模式 | `ray.init(address='ray://...')` | 远程连接。连接到远程服务器上的 Ray 集群。代码在本地跑，但任务在远程集群执行。 | 开发机提交任务到云端/服务器集群。 |


3. 关键参数详解

虽然 ray.init() 有很多参数，但在实际工程中，最常用的主要有以下几个：
- address (str):
    - 指定要连接的 Ray 集群地址。
    - 设为 'auto'：自动连接本机现有的集群。
    - 设为 'ray://<IP>:10001'：连接远程集群（Ray Client 模式）。
    - 设为 None（默认）：<b>尝试连接现有集群，如果找不到，则启动一个新的本地集群。</b>
- num_cpus (int):
    - 限制 Ray 使用多少个 CPU 核心。
    - <b>如果不传，Ray 默认会使用机器上所有的 CPU 核心。</b>在开发时，为了防止电脑卡死，可以手动限制（例如 num_cpus=4）。
- num_gpus (int):
    - 限制 Ray 使用多少个 GPU。
    - Ray 不会自动检测 GPU 显存，它只管理 GPU 的“槽位”。
- <font color='red'>runtime_env (dict):</font>
    - <font color='red'>非常重要。用于定义任务运行时的环境依赖。</font>
    - <b>你可以指定 pip 包列表、working_dir（工作目录）或环境变量。Ray 会自动将这些依赖同步到集群的 Worker 节点上，确保环境一致。</b>
- namespace (str):
    - <b>用于隔离不同的任务或用户。类似于 Kubernetes 的 Namespace，防止不同任务之间的命名冲突。</b>
    
4. 代码实战示例
    
##### 场景 A：本地开发（最简单）
    
在 Jupyter Notebook 或本地脚本中调试：

In [ ]:
import ray

# 启动本地集群，限制使用 4 个 CPU 核心，防止电脑卡死
ray.init(num_cpus=4, ignore_reinit_error=True)

@ray.remote
def f(x):
    return x * x

# 提交任务
futures = [f.remote(i) for i in range(4)]
# 获取结果
print(ray.get(futures))

# 结束后关闭
ray.shutdown()

##### 场景 B：连接生产集群（带环境依赖）

当你把代码提交到服务器集群运行时：

In [ ]:
import ray

# 连接到远程集群，并指定运行环境
ray.init(
    address="ray://192.168.1.100:10001", 
    runtime_env={
        "pip": ["requests", "torch"],  # 自动安装依赖
        "working_dir": "./my_project"  # 自动上传代码
    },
    namespace="training_job_01"
)

5. 避坑指南

    1. 重复初始化：在同一个 Python 进程中多次调用 ray.init() 会报错。通常加上 ignore_reinit_error=True 参数来避免这个问题（特别是在 Jupyter 中反复运行单元格时）。
    2. 僵尸进程：如果脚本异常退出，Ray 的后台进程可能不会自动关闭，导致下次启动失败。可以使用 ray.shutdown() 或在命令行使用 ray stop 来清理。
    3. <b>资源死锁：如果你限制了 num_cpus=2，但启动了 3 个需要 CPU 的 Actor，第 3 个 Actor 会一直等待直到有资源释放，这可能导致死锁。</b>
    
总结来说，ray.init() 是你进入 Ray 分布式世界的大门，它负责把底层的复杂性（进程管理、通信、资源分配）封装起来，让你能以极简的方式开始并行编程。

## Worker初始化（trainer.init_workers()） 使用ray后端初始化分布式训练workers

- 动作：ray在各个节点上启动远程任务（ray actors）
    - 根据配置创建ray资源池
    - 为每个角色（actor，critic等）创建worker组
- 动态导入： 根据配置中的strategy（如fsdp），动态导入对应的Worker类（如：FSDPWroker）
- 模型加载：
    - 在各个worker内部，根据actor_rollout_ref.model.path加载Qwen2.5-32B模型权重
    - 初始化FSDP进程组，将模型分片到不同GPU上

### trainer.init_workers() 是 VeRL 框架中从“单兵作战”转向“集团军作战”的关键转折点。

如果说 trainer 是总指挥，那么 init_workers() 就是“点兵点将”的过程。在这一步之前，代码只是在本地定义了一些配置和类；在这一步之后，真正的分布式计算资源（GPU 进程）被启动，各个角色（Actor, Critic, Reward）各就各位，准备开始协同工作。

#### 以下是 trainer.init_workers() 的详细执行逻辑拆解：

##### 🎯 核心目标

init_workers() 的核心任务只有两个：
 - 资源分配：告诉 Ray 我需要多少 GPU，怎么分组（资源池）。
 - 进程启动：在分配好的 GPU 上启动具体的 Worker 进程，并初始化模型。
 
⚙️ 执行流程三步走

这个过程通常发生在 RayPPOTrainer 内部，具体逻辑如下：

1. 创建资源池 (Create Resource Pools)
首先，Trainer 会根据配置文件（config），向 Ray 申请资源。<b>VeRL 使用 ResourcePoolManager 来管理这些资源。</b>
- 逻辑：系统会定义不同的“角色组”，例如 ActorRollout 需要 4 张卡，Critic 需要 2 张卡。
- 代码行为：

In [ ]:
trainer = RayDAPOTrainer(
            config=config,
            tokenizer=tokenizer,
            processor=processor,
            role_worker_mapping=role_worker_mapping,
            resource_pool_manager=resource_pool_manager,
            ray_worker_group_cls=ray_worker_group_cls,
            reward_fn=reward_fn,
            val_reward_fn=val_reward_fn,
)
trainer.init_workers()  # initialize workers
trainer.fit()

In [ ]:
# 伪代码：创建资源池
# 比如定义一个包含 4 个 GPU 的全局资源池
# 这一步只是“圈地”，还没有启动进程。
self.resource_pool_manager.create_resource_pool()

2. 映射角色与类 (Map Roles to Classes)

接下来，Trainer 需要决定哪个资源池运行什么代码。VeRL 支持 FSDP 和 Megatron 两种后端，这里会根据配置动态选择 Worker 类。

- 逻辑：
    - 如果配置是 fsdp，则选择 ActorRolloutRefWorker (FSDP版)。
    - 如果配置是 megatron，则选择对应的 Megatron Worker。
- 代码行为：

In [ ]:
self.resource_pool_manager.create_resource_pool() # 创建资源池

self.resource_pool_to_cls = {
    pool: {} for pool in self.resource_pool_manager.resource_pool_dict.values()
} # 创建资源池到类映射

# create actor and rollout
if self.hybrid_engine:
    resource_pool = self.resource_pool_manager.get_resource_pool(
        Role.ActorRollout
    )  # 获取资源池
    actor_rollout_cls = RayClassWithInitArgs(
      cls=self.role_worker_mapping[Role.ActorRollout],
      config=self.config.actor_rollout_ref,
      role="actor_rollout",
      profile_option=self.config.trainer.npu_profile.options,
    ) # 创建actor和rollout
    self.resource_pool_to_cls[resource_pool][
        "actor_rollout"
    ] = actor_rollout_cls
else:
    raise NotImplementedError

这里使用了 RayClassWithInitArgs，这是一个包装器，允许我们在远程启动时传递初始化参数（如模型路径、超参数）。

RayClassWithInitArgs 是一个用于在 Ray 框架中延迟启动 Actor 的辅助类，其作用类似于一个工厂类。它封装了 Actor 的定义和初始化参数，使得 Actor 的实际创建可以被推迟到需要时再进行。

###### 🎯 主要功能

RayClassWithInitArgs 的核心功能是在调用时才真正启动一个 Ray Actor。它继承自 ClassWithInitArgs，主要完成了以下两件事：

1. 保存 Actor 定义和初始化参数：在实例化 RayClassWithInitArgs 时，它会接收并保存一个通过 @ray.remote 装饰的 Actor 类（cls）以及用于初始化该 Actor 的参数（*args, **kwargs）。此时，Actor 并不会被立即创建。

2. 提供带策略的 Actor 创建方法：<font color='red'>当调用 RayClassWithInitArgs 的实例时（即执行 __call__ 方法），它会使用之前保存的 Actor 类和参数，并结合调度策略来异步地创建一个 Actor 实例。</font>


###### ⚙️ 工作流程

RayClassWithInitArgs 的工作流程可以概括为以下几个步骤：

1. 初始化：创建 RayClassWithInitArgs 实例，传入目标 Actor 类和其初始化参数。
2. 配置调度选项：在调用时，通过 __call__ 方法传入 placement_group 等参数，构建 Actor 的调度策略 options。这决定了 Actor 将在哪些资源上启动。
3. 异步启动 Actor：最后，使用 self.cls.options(**options).remote(*self.args, **self.kwargs) 来异步地启动 Actor。
    - .options(**options) 用于设置调度策略，而 .remote(...) 则是 Ray 中用于异步创建 Actor 或调用任务的方法。
    
<font color='red'>简而言之，RayClassWithInitArgs 将 Actor 的“定义”与“创建”两个动作分离开来，为在复杂分布式环境中灵活、按需地管理 Actor 生命周期提供了便利。</font>

In [ ]:
class RayClassWithInitArgs(ClassWithInitArgs):
    """A wrapper class for Ray actors with initialization arguments.

    This class extends ClassWithInitArgs to provide additional functionality for
    configuring and creating Ray actors with specific resource requirements and
    scheduling strategies.
    """

    def __init__(self, cls, *args, **kwargs) -> None:
        # self._options = kwargs.pop('options', dict())
        super().__init__(cls, *args, **kwargs)
        self._options = {}
        self._additional_resource = {}

    def set_additional_resource(self, additional_resource):
        """Set additional resource requirements for the actor.

        Args:
            additional_resource: Dictionary specifying additional resource requirements
        """
        self._additional_resource = additional_resource

    def update_options(self, options: dict):
        """Update the Ray actor creation options.

        Args:
            options: Dictionary of options to update
        """
        self._options.update(optionsi)

    
    
    # 使用 call将定义和创建两个动作分离开来
    def __call__(
        self,
        placement_group,
        placement_group_bundle_idx,
        use_gpu: bool = True,
        num_gpus=1,
        sharing_with=None,
        device_name="cuda",
    ) -> Any:
        """Create and return a Ray actor with the configured options.

        Args:
            placement_group: Ray placement group for scheduling
            placement_group_bundle_idx: Index of the bundle in the placement group
            use_gpu: Whether to use GPU resources
            num_gpus: Number of GPUs to allocate
            sharing_with: Actor to share resources with
            device_name: Device for training

        Returns:
            A Ray actor handle with the configured options
        """
        if sharing_with is not None:
            target_node_id = ray.get(sharing_with.get_node_id.remote())
            visible_devices = ray.get(sharing_with.get_cuda_visible_devices.remote())
            options = {"scheduling_strategy": NodeAffinitySchedulingStrategy(node_id=target_node_id, soft=False)}
            return self.cls.options(**options).remote(*self.args, cuda_visible_devices=visible_devices, **self.kwargs)

        options = {
            "scheduling_strategy": PlacementGroupSchedulingStrategy(
                placement_group=placement_group, placement_group_bundle_index=placement_group_bundle_idx
            )
        }
        options.update(self._options)

        if use_gpu and device_name == "cuda":
            options["num_gpus"] = num_gpus
        if use_gpu and device_name == "npu":
            options["resources"] = {"NPU": num_gpus}

        if len(self._additional_resource) > 1:
            for k, v in self._additional_resource.items():
                options[k] = v

        # print("cls:", self.cls)
        # print("args: ", self.args)
        # print("kwargs: ", self.kwargs)
        return self.cls.options(**options).remote(*self.args, **self.kwargs)


###### <font color='red'> \_\_call\_\_ </font>

这个 __call__ 方法是 RayClassWithInitArgs 类的核心，<font color='red'>它负责根据传入的参数和内部配置，灵活地创建并返回一个 Ray Actor 实例。</font>

整个方法的逻辑可以分为两大分支：
- 与指定 Actor 共享资源，
- 或根据放置组（Placement Group）进行调度。

###### 🤝 资源亲和性与共享模式

当 sharing_with 参数不为 None 时，此方法会进入资源共享模式。<font color='red'>它的主要目标是确保新创建的 Actor 与 sharing_with 所代表的 Actor 运行在同一个物理节点上，并共享其 GPU 资源。</font>

1. 获取目标节点信息：
    - 首先，通过 ray.get(sharing_with.get_node_id.remote()) 获取 sharing_with Actor 所在节点的 ID。
    - 然后，通过 ray.get(sharing_with.get_cuda_visible_devices.remote()) 获取该 Actor 可见的 CUDA 设备列表。
2. 配置节点亲和性策略：
    - 使用 NodeAffinitySchedulingStrategy 创建一个调度策略，并将 node_id 设置为目标节点的 ID。soft=False 表示这是一个硬性要求，新 Actor 必须被调度到该节点上。
3. 创建 Actor：
    - 最后，使用配置好的 options 和获取到的 cuda_visible_devices 来启动新的 Actor。这确保了新 Actor 不仅在同一节点，而且能正确识别和使用相同的 GPU 设备。

In [ ]:
# 资源亲和性模式下的核心逻辑
if sharing_with is not None:
    # 1. 获取目标 Actor 的节点ID和可见的GPU设备
    target_node_id = ray.get(sharing_with.get_node_id.remote())
    visible_devices = ray.get(sharing_with.get_cuda_visible_devices.remote())
    
    # 2. 设置节点亲和性调度策略
    options = {"scheduling_strategy": NodeAffinitySchedulingStrategy(node_id=target_node_id, soft=False)}
    
    # 3. 启动Actor，并指定其可见的GPU设备
    return self.cls.options(**options).remote(*self.args, cuda_visible_devices=visible_devices, **self.kwargs)

###### 🗺️ 基于放置组的调度模式

这是 __call__ 方法的默认行为，当 sharing_with 为 None 时触发。它利用 Ray 的放置组（Placement Group）来精确控制 Actor 的资源分配和位置。

1. 配置放置组策略：
    - 创建一个 PlacementGroupSchedulingStrategy，<font color='red'>指定 Actor 应该被放置在哪个 placement_group 以及组内的哪个资源束（placement_group_bundle_idx）上。</font>
2. 合并其他选项：
    - options.update(self._options)：将实例内部可能预先设置的其他选项合并进来。
    - 设置计算资源：根据 use_gpu 和 device_name 参数，决定分配 GPU 还是 NPU（华为昇腾处理器）资源。
        - 如果使用 CUDA GPU，则设置 options["num_gpus"]。
        - 如果使用 NPU，则设置 options["resources"] = {"NPU": num_gpus}。
    - 设置额外资源：如果 self._additional_resource 字典中包含其他资源需求（如自定义资源），则将它们也添加到 options 中。
3. 创建 Actor：
    - <font color='red'>最终，使用整合了所有资源和调度策略的 options 来启动 Actor。</font>

In [ ]:
# 基于放置组调度模式下的核心逻辑
# 1. 设置基础的放置组调度策略
options = {
    "scheduling_strategy": PlacementGroupSchedulingStrategy(
        placement_group=placement_group, placement_group_bundle_index=placement_group_bundle_idx
    )
}

# 2. 合并其他预设选项
options.update(self._options)

# 3. 根据设备类型设置计算资源
if use_gpu and device_name == "cuda":
    options["num_gpus"] = num_gpus
if use_gpu and device_name == "npu":
    options["resources"] = {"NPU": num_gpus}

# 4. 添加任何额外的资源需求
if len(self._additional_resource) > 1:
    for k, v in self._additional_resource.items():
        options[k] = v

# 5. 启动Actor
return self.cls.options(**options).remote(*self.args, **self.kwargs)

3. 实例化 WorkerGroup (Instantiate Worker Groups)

这是真正的“点火启动”时刻。<b>Trainer 调用 RayWorkerGroup，Ray 会根据之前定义的资源池和类，在指定的 GPU 上启动进程。</b>

- 逻辑：
    - Actor Group：<b>启动负责生成数据和训练策略的 Worker。</b>
    - Critic Group：<b>启动负责价值估计的 Worker。</b>
    - Reward/Ref Group：<b>启动负责打分和计算 KL 散度的 Worker。</b>
- 代码行为：

In [ ]:
# 伪代码：实际启动
# 这行代码会触发 Ray 远程执行 __init__ 方法
self.actor_rollout_wg = RayWorkerGroup(
    resource_pool=self.resource_pool_manager.get_resource_pool(Role.ActorRollout),
    cls_with_init_args=actor_rollout_cls
)



<font color='red'>一旦这行代码执行完毕，底层的 GPU 进程就已经运行起来了，模型权重也已经被加载到了显存中。</font>

🔍 关键细节与“坑”

在 init_workers() 的执行过程中，有几个非常值得注意的技术细节：

- Hybrid Engine 的体现：

<font color='red'>在 VeRL 中，ActorRolloutRefWorker 通常是一个混合体。它既包含用于训练的 FSDP 模型，也包含用于生成的 vLLM 引擎。init_workers() 会同时初始化这两部分（虽然 vLLM 引擎可能是在第一次生成时才真正加载权重，但进程已经准备好了）。</font>

- Rank 的分配：

在分布式训练中，每个进程都有一个 rank ID。init_workers() 会确保每个 WorkerGroup 内部的进程拥有正确的 rank（0, 1, 2...），这对于后续的 all_reduce 通信至关重要。

- ###### 单进程控制，多进程计算：

init_workers() 是在主进程（Driver Process）中同步调用的，但它启动的是远程进程。<font color='red'>这意味着，调用完成后，主进程依然是单线程的控制器，而被启动的 Workers 则是独立的计算实体。</font>

# 第三阶段 ： 训练循环-trainer.fit()


trainer.fit() 是 VeRL 训练流程的绝对核心，也是整个强化学习（RL）循环的“心脏”。

- 如果说 trainer.init_workers() 是把演员（Actor）、评论家（Critic）等角色请上舞台，那么 trainer.fit() 就是正式开演。<font color='red'>它负责编排整个 PPO（或 GRPO）算法的执行顺序，控制数据在各个 Worker 之间流转，并驱动模型参数的更新。</font>

- 在 VeRL 的实现中，fit() 函数通常位于 RayPPOTrainer 类中。它的核心逻辑是一个多层嵌套的循环，我们将这个循环拆解为以下几个关键阶段来详细介绍：


🎬 整体架构：三层循环

fit() 的结构通常如下所示：

1. 最外层：Epoch 循环 —— 控制整个数据集被训练多少轮。
2. 中间层：Step/Batch 循环 —— <font color='red'>按批次处理数据，这是 PPO 算法的主要迭代单位。</font>
3. 最内层：PPO 更新循环 —— <b>在同一个批次数据上进行多次（ppo_epochs）梯度下降更新。</b>

🔄 核心流水线：一个 PPO Step 的一生

在中间层的每一次迭代（Step）中，trainer.fit() 会像流水线一样执行以下 5 个关键步骤。这是 VeRL 最精彩的部分，数据在这里被不断加工。

### 第一步：生成经验 (Rollout / Generation)

这是 RL 区别于监督学习的地方。模型必须先“尝试”生成答案，才能知道好坏。

- 动作：Trainer 从数据加载器中取出一批提示词（Prompts），发送给 Actor Worker。
- <b>调用：</b> ActorWorker.generate_sequences(prompts)
- 底层逻辑：
    - 读取 data.train_files
    - Actor Worker <font color='red'>切换到推理模式（通常使用 vLLM 引擎）</font>。
        - 使用vLLM(配置中指定 rollout.name=vllm) 进行推理
    - 利用当前的策略网络 $π_{θ_{old}}$ 为每个 Prompt 生成 N 个回复（Responses）。
- 数据流向：Prompts →→ Actor →→ Prompts + Responses。
- 输出： 获得(promt,response)对

#### ActorWorker.generate_sequences(prompts)

- 它的任务非常明确：利用当前的策略模型（Actor），针对输入的 Prompts，快速、高效地生成高质量的回复序列（Responses）。
- 在 VeRL 的混合架构（Hybrid Engine）下，这个方法不仅仅是调用一下模型，<font color='red'>它背后隐藏了一套复杂的“变身”与“同步”机制。</font>

以下是 actor_worker.generate_sequences() 的详细执行逻辑拆解：

##### 🎯 核心功能：它做什么？

<font color='red'>简单来说，它接收一批提示词（Prompts），输出对应的生成结果（Responses）以及生成过程中的元数据（如 Attention Mask, Position IDs）。</font>

但在 VeRL 中，这个过程有三个关键特征：

1. <font color='red'>多副本生成 (Rollout)</font>：对于 GRPO 或 PPO 算法，它通常不是生成 1 个回复，而是为每个 Prompt 生成 N 个回复（例如 N=5 或 N=12 ）。
2. <font color='red'>引擎切换：它需要在“训练模式”（FSDP/Megatron）和“推理模式”（vLLM/SGLang）之间无缝切换。VeRL 实现了Hybrid Engine，让训练和推理共用一套显存，既保证了训练的高效（FSDP），又保证了推理的极速（vLLM），且无需手动管理复杂的权重转换。</font>
3. 分布式执行：它会在多个 GPU 上并行生成，最后聚合结果。

##### ⚙️ 执行流程：四步走

<font color='red'>当 trainer 调用 actor_worker.generate_sequences(prompts) 时，底层会发生以下四个步骤：</font>

###### 第一步：权重同步 (Weight Sync) —— 最关键的“幕后英雄”

这是 VeRL 最核心的设计之一。

- <font color='red'>背景：Actor Worker 平时处于“训练模式”（FSDP 状态），模型参数被切分（Sharding）在多个 GPU 上。但生成任务需要“推理模式”（vLLM 状态），vLLM 需要完整的、未切分的模型权重。</font>
- 动作：
    1. 进入 generate_sequences 方法时，首先会触发 <font color='red'>rollout_sharding_manager 的上下文管理器（__enter__）。</font>
    2. <font color='red'>FSDP 后端：</font>将切分在各地的参数通过 all_gather 收集起来，转换成 vLLM 可用的格式，并加载到 vLLM 引擎中。
    3. <font color='red'>Megatron 后端：</font>将参数从训练引擎“搬运”到推理引擎。
- 意义：确保推理引擎使用的是刚刚更新过的最新策略，而不是旧版本。


###### 第二步：数据预处理 (Data Preprocessing)
- 动作：
    - 将输入的 DataProto（包含 Input IDs, Attention Mask 等）转换为推理引擎能理解的格式。
    - <font color='red'>重复数据：如果配置了 rollout.n = 5，这里会将每个 Prompt 复制 5 份，以便一次性生成 5 个不同的回复。</font>
    - 分片：根据 Tensor Parallelism (TP) 的设置，将数据切分给不同的 GPU。
    
###### 第三步：调用推理引擎 (Inference Execution)
- 动作：
    - 正式调用底层的推理引擎（通常是 vLLMRollout 或 SGLangRollout）。
    - 利用 Continuous Batching 和 PagedAttention 技术，在 GPU 上高效地生成 Token。
    - <font color='red'>应用采样策略（如 Temperature, Top-P, Top-K）来保证生成的多样性。</font>
- 输出：生成 Token 序列、每个 Token 的对数概率（Log Probs，用于后续计算重要性采样比率）。

###### 第四步：后处理与恢复 (Postprocessing & Restore)
- 动作：
    1. <font color='red'>聚合结果：将分散在不同 GPU 上的生成结果收集起来（all_gather）。</font>
    2. 格式转换：将结果重新封装回 VeRL 的 DataProto 格式。
    3. <font color='red'>权重恢复：退出上下文管理器（__enter__ 对应的 __exit__），将模型状态从“推理模式”切回“训练模式”，释放 vLLM 占用的显存，准备进行下一轮的梯度更新。</font>
    
    
在 verl/workers/fsdp_workers.py 中，逻辑大致如下：

In [ ]:
@register(dispatch_mode=Dispatch.DP_COMPUTE_PROTO) # 分布式注册
def generate_sequences(self, prompts: DataProto):
    # 1. 数据移到 GPU
    prompts = prompts.to('cuda')
    
    # 2. 上下文管理器：负责权重的同步与恢复
    # 这里是“变身”的关键时刻
    with self.rollout_sharding_manager:
        
        # 3. 预处理：比如重复 N 次 (rollout.n)
        # rollout_sharding_manager 的上下文管理器（__enter__）
        prompts = self.rollout_sharding_manager.preprocess_data(prompts)
        
        # 4. 核心生成：调用 vLLM/SGLang的generate_sequences方法
        # 此时模型已经是 vLLM 格式，且权重是最新的
        output = self.rollout.generate_sequences(prompts=prompts)
        
        # 5. 后处理：聚合结果
        output = self.rollout_sharding_manager.postprocess_data(output)
        
    return output

In [ ]:
 with self.rollout_sharding_manager:
            log_gpu_memory_usage("After entering rollout sharding manager", logger=logger)

            with simple_timer("generate_sequences", timing_generate):
                output = self.rollout.generate_sequences(prompts=prompts) # 调用 vLLM/SGLang的generate_sequences方法

            log_gpu_memory_usage("After rollout generation", logger=logger)

######  📌 <font color='red'>with self.rollout_sharding_manager: 是 VeRL 框架中实现 Hybrid Engine（混合引擎） 的“魔法开关”。</font>

<font color='red'>你可以把它想象成一个“变形金刚的变身指令”。在这个 with 代码块内部，模型是推理模式（vLLM/SGLang）；出了这个代码块，模型又变回了训练模式（FSDP/Megatron）。</font>

<b>这行代码虽然简短，但它在后台极其高效地解决了强化学习中最大的痛点：如何在训练权重（训练模式）和生成数据（推理模式）之间无缝切换，且不浪费显存。</b>

###### 🛠️ 核心身份：Python 上下文管理器

从 Python 语法角度看，rollout_sharding_manager 实现了一个上下文管理器（Context Manager）。这意味着它定义了“进入”和“退出”时的行为：

- __enter__ (进入时)：变身推理模式。负责把训练用的模型权重“搬运”并“重组”成推理引擎（vLLM）能用的格式。
- __exit__ (退出时)：恢复训练模式。负责把推理引擎占用的显存释放掉，或者把控制权交还给训练引擎，准备进行梯度更新。


###### ⚙️ 执行流程：三步走

当代码执行到 with self.rollout_sharding_manager: 时，具体发生了以下事情：

###### 第一步：进入上下文 (__enter__) —— 权重同步

这是最关键的一步。此时，Actor 模型正处于 FSDP 训练状态，它的参数是被切分（Sharded）在多个 GPU 上的，且包含梯度状态。但 vLLM 推理引擎需要完整的、未切分的模型权重。

1. 收集权重 (All-Gather)：
    - 管理器会触发分布式通信操作（如 all_gather），将所有 GPU 上切分的 FSDP 参数收集起来，<font color='red'>拼成完整的模型权重。</font>
2. 格式转换与加载：
    - 将拼好的完整权重，直接加载到 vLLM 引擎 的显存空间中。
    - 注意：在 VeRL 的 FSDP 后端中，这通常通过 FSDPVLLMShardingManager 实现，它会处理 PyTorch Tensor 到 vLLM 内部 Tensor 的转换。
3. 唤醒推理引擎：
    - 如果 vLLM 处于“休眠模式”（为了节省显存），管理器会调用 wake_up() 将其激活。
    
###### 结果：此时，vLLM 引擎手里拿着和 Actor 训练模型一模一样的最新权重，准备好进行生成了。


###### 第二步：执行生成 (Inference)

在 with 代码块内部，代码会调用 self.rollout.generate_sequences(prompts)。
- 此时，请求被发往 vLLM。
- vLLM 利用刚才同步过来的权重，高效地生成回复（Responses）。
- 由于 vLLM 使用了 PagedAttention 和 Continuous Batching，这一步非常快。


###### 第三步：退出上下文 (__exit__) —— 资源回收

当 with 代码块执行完毕（生成结束），管理器会自动执行清理工作：

1. 释放推理显存：
    - 调用 vLLM 的 sleep() 或 reset_prefix_cache()。
    - 释放掉 vLLM 占用的 KV Cache 和模型权重显存。这是为了给接下来的梯度计算和优化器更新腾出宝贵的显存空间。
2. 恢复训练状态：
    - 控制权回到 FSDP 模型。此时 FSDP 模型依然是完整的（或者重新切分），且处于“训练模式”，随时准备接收梯度更新。
    
    
###### 📊 为什么要这么做？（解决了什么问题）

你可能会问：为什么不直接让训练模型去生成？或者一直保留两个模型？

1. 避免 OOM (显存爆炸)：
    - 如果同时保留一份完整的 FSDP 训练模型和一份完整的 vLLM 推理模型，显存占用会翻倍，很容易导致 OOM。
    - rollout_sharding_manager 通过<font color='red'>分时复用显存：训练时只留训练模型，生成时临时加载推理模型，生成完立刻释放。</font>
2. 兼顾速度与精度：
    - 训练时：用 FSDP，因为它支持大规模分布式训练，计算梯度快。
    - 生成时：用 vLLM，因为它针对推理做了极致优化（如 FlashAttention），生成速度快。
    - <font color='red'>这个 with 语句就是连接这两个世界的桥梁。</font>
3. 保证权重一致性：
    - 在强化学习中，生成数据必须基于当前最新的策略。这个管理器确保了每次生成前，vLLM 都会强制同步 Actor 的最新权重，避免了“用旧模型生成数据”的严重错误。
    
在 VeRL（以及大多数 RLHF 框架）中，vLLM 的核心任务就是利用刚刚更新过的最新权重来生成数据，以便评估新策略的效果。

但是，vLLM 作为一个推理引擎，它的权重不会“自动”变过来。在 VeRL 的架构中，这背后有一个“显式同步”的过程。

为了让你彻底明白这个机制，我们需要区分“训练权重”和“推理权重”，并看看它们是如何“握手”的：


###### 1. 两个世界的权重
在 VeRL 的 Hybrid Engine 中，存在两种状态的模型：

- 训练模式 (FSDP/Megatron)：
    - 这是“大脑”的学习中心。
    - 权重是被切分（Sharded）在多个 GPU 上的，且时刻伴随着梯度的计算和更新。
    - 状态：时刻在变（正在学习）。
- 推理模式 (vLLM)：
    - 这是“嘴巴”的发声器。
    - 权重需要是完整的（Unsharded）且连续的，为了追求极致的生成速度，它通常驻留在显存的特定区域。
    - 状态：静止不动（除非有人喂给它新权重）。
    
###### 为什么要这么做？
你可能会问：为什么不直接让 vLLM 参与训练？或者让 FSDP 直接生成？

- FSDP 生成太慢：FSDP 是为了训练设计的，每次生成一个 token 都需要进行复杂的跨卡通信来拼凑权重，速度极慢。
- vLLM 生成极快：vLLM 专门为推理优化，利用连续批处理和分页注意力，吞吐量比 FSDP 高几十倍。
- 必须用新权重：强化学习（RL）的核心逻辑是“探索-评估-更新”。
    - 如果 vLLM 用的是旧权重，生成的回复就代表不了当前的策略，计算出的优势（Advantage）就是错的，训练就会崩塌。
    - 所以，必须确保 vLLM 生成时，使用的是 Actor 刚刚更新完的权重。
    
###### 📌 总结

在 VeRL 的每一个 PPO 迭代步中：
1. Actor (FSDP) 先更新参数（变强了）。
2. ShardingManager 把变强后的参数搬运给 vLLM。
3. vLLM 带着新参数去生成数据（去探索）。

### 第二步：计算价值与概率 (Compute Values & Log Probs)

生成了回复后，我们需要计算“旧策略”认为这些回复有多好，以及“评论家”认为这些状态值多少分。
- 动作：

    1. 计算 Log Prob：Actor Worker（切换回训练模式）和 Reference Worker <font color='red'>分别计算生成序列的对数概率。</font> 这是为了后续计算 KL 散度（惩罚模型偏离原始分布太远）。
        - 调用：ReferenceWorker.compute_log_probs(prompts,responses)
            - 加载Reference Model(通常是初始模型)
            - 计算生成这些回复的对数概率$logπ_{ref}(y|x)$
    2. 计算价值 (Values)：Trainer 将序列发送给 Critic Worker。Critic 网络会对序列中的每个 Token 打分，预测其未来的期望回报 V(s) 。
        

- 数据流向：Sequences →→ Actor/Ref/Critic →→ Log Probs + Values。
  

####  ReferenceWorker.compute_log_probs(prompts, responses) 是 VeRL 训练循环中的“定海神针”。

如果说 Actor 是不断尝试新事物的“探险家”，那么 Reference Worker 就是那个“守旧的老学究”。它的任务不是生成新内容，而是“背诵”——拿着 Actor 生成的答案，去计算在“初始模型”眼里，这些答案出现的概率是多少。

这个函数最核心的作用是为 PPO/GRPO 算法提供 KL 散度（KL Divergence） 计算所需的基准数据，防止模型在训练过程中“放飞自我”，导致灾难性遗忘。

以下是 ReferenceWorker.compute_log_probs() 的详细执行逻辑拆解：

##### 🎯 核心功能：它做什么？

简单来说，它接收 prompts 和 responses（由 Actor 刚刚生成的），输出这些序列在 Reference Model（参考模型） 下的对数概率（Log Probs）。

- 输入：prompts + responses（完整的序列）。
- 输出：ref_log_probs（每个 Token 在参考模型眼里的概率分布）。
- 关键点：<b>Reference Model 的权重是冻结（Frozen）的，它通常就是 RLHF 训练开始前的那个 SFT（监督微调）模型。</b>

##### ⚙️ 执行流程：三步走

当 trainer 调用这个函数时，底层会发生以下步骤：

######  第一步：上下文管理器与权重保护

Reference Worker 通常不需要像 Actor 那样频繁更新权重。
- 动作：
    - 进入 compute_log_probs 时，通常会包裹在一个上下文管理器中（类似于 Actor 的 rollout 管理器，但更简单）。
    - 无需同步：与 Actor 不同，Reference Worker 不需要从外部同步权重。它的权重在初始化时加载一次（或者从 Actor 的初始权重复制一份），之后永远保持不变。
    - <b>模式切换：将模型设置为 eval() 模式，并关闭梯度计算（torch.no_grad()），因为它只负责推理，不负责训练。</b>
    
    
###### 第二步：前向传播 (Forward Pass)

这是计算的核心。注意，这里不是生成（Generation），而是前向计算。

- 数据组装：
    - 将 prompts 和 responses 拼接成完整的输入序列 input_ids。
    - 构建对应的 attention_mask。
- 模型推理：
    - 将完整的序列输入到 Reference Model 中。
    - 模型输出 logits（词汇表大小的向量）。
- 代码逻辑示意：



In [ ]:
with torch.no_grad(): # 绝不计算梯度
    # 1. 拼接 Prompt 和 Response
    input_ids = torch.cat([prompts, responses], dim=-1)
    
    # 2. 送入 Reference Model
    # 此时模型权重是冻结的 SFT 模型
    outputs = self.ref_model(input_ids, attention_mask=mask)
    
    # 3. 获取 Logits
    logits = outputs.logits

###### 第三步：计算 Log Probs (Loss Calculation)

拿到 logits 后，需要计算生成这些特定 Token 的概率。

- 动作：
    - Shift：将 logits 向左移动一位（因为模型是预测下一个 token）。
    - Gather：使用 responses 的 token ids 作为索引，从 logits 中提取对应位置的概率值。
    - Log Softmax：将提取的值转换为对数概率。
- 结果：得到一个与 responses 长度相同的张量，代表 Reference Model 认为这些词出现的“合理性”。

在 verl/workers/reference_worker.py 中，逻辑大致如下

In [ ]:
def compute_log_probs(self, batch):
    # 1. 提取数据
    input_ids = batch['input_ids'] # 包含 prompt + response
    attention_mask = batch['attention_mask']
    response_length = batch['response_length']
    
    # 2. 进入无梯度模式 (推理)
    with torch.no_grad():
        # 3. 前向传播
        # 这里的 self.model 是冻结的 Reference Model
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False # 推理不需要缓存
        )
        
        # 4. 计算 Log Probs
        # 使用 logits 计算每个 token 的对数概率
        log_probs = self._compute_log_probs_from_logits(
            outputs.logits, 
            input_ids, 
            response_length
        )
        
    return log_probs

🌟 总结
ReferenceWorker.compute_log_probs() 是 VeRL 训练中的“质检员”。

- 它不参与梯度的反向传播（权重冻结）。
- 它不生成任何新内容（只做前向计算）。
- 它只提供一个基准概率，用来计算 Actor 偏离原始模型有多远。

### 第三步：计算奖励 (Compute Rewards)

这一步是告诉模型“什么是对的”。
- 动作：
    - Trainer 调用 Reward Worker（或本地的 Reward Function）。
        - 调用RewardWorker.compute_rewards(prompts,responses)
            - 加载reward_model.reward_mannager=dapo
            - 执行数学规则校验（检查答案是否正确）
            - 应用overlong_buffer惩罚（如果回复太长）
    - 根据规则（如 GSM8K 的答案正确性）或奖励模型（Reward Model）的打分，为生成的序列计算最终奖励 r 。
- 数据流向：Sequences →→ Reward Model →→ Rewards。
    -   - 输出：每个回复的奖励分数R

#### RewardWorker.compute_rewards(prompts, responses) 是 VeRL 训练循环中的“裁判”环节。

如果说 generate_sequences 是让学生（Actor）交卷，那么 compute_rewards 就是老师批改作业。它的核心任务非常明确：接收模型生成的回复（Responses），根据某种标准给出一个分数（Reward），告诉模型刚才生成得好不好。

在 VeRL 的架构中，这个函数不仅仅是简单的“打分”，它还涉及到分布式数据组装、多策略选择（规则 vs 模型）以及Token 级奖励映射。

以下是 RewardWorker.compute_rewards() 的详细执行逻辑拆解：

##### 🎯 核心功能：它做什么？

<font color='red'>简单来说，它接收一批 prompts 和 responses，输出一个标量分数（或张量）。</font>但在 VeRL 中，这个过程有三种主要的实现路径，取决于你的训练目标：

1. 规则打分 (Rule-based)：用于数学（GSM8K）、代码（HumanEval）等任务。检查答案是否正确、代码能否运行。
2. 模型打分 (Model-based)：用于 RLHF（人类反馈强化学习）。加载一个专门的 Reward Model (RM) 来模拟人类偏好。
3. 混合打分：结合两者，比如“答案正确 + 格式规范”。

#####  ⚙️ 执行流程：四步走

当 trainer 调用 reward_worker.compute_rewards(prompts, responses) 时，底层会发生以下步骤：

###### 第一步：数据组装与预处理 (Data Assembly)

<font color='red'>Reward Model 或打分函数通常需要一个完整的对话上下文，而不仅仅是回复。</font>
- 动作：
    - 将 prompts 和 responses 拼接起来，形成完整的序列：[Prompt] + [Response]。
    - 进行 Tokenization（如果是模型打分），将其转换为 input_ids 和 attention_mask。
    - 分片：根据分布式策略，将数据切分给不同的 GPU（如果是多卡推理）。
    
######  第二步：选择打分策略 (Dispatch Strategy)

VeRL 非常灵活，compute_rewards 会根据配置决定走哪条路：

- 路径 A：函数型打分 (Function-based)
    - 场景：数学题、代码题。
    - 逻辑：
        1. 调用 Python 函数（如 check_answer(response)）。
        2. 数学：用正则表达式提取 #### 42，对比标准答案。
        3. 代码：把生成的代码丢进沙箱（Sandbox）跑测试用例，看通过率。
    - 特点：极快，无需 GPU，准确无误。
- 路径 B：模型型打分 (Model-based)
    - 场景：对话对齐、风格迁移。
    - 逻辑：
        1. 调用 Reward Model (RM) 的推理引擎（通常也是 vLLM 或 HF Transformers）。
        2. 将拼接好的序列送入 RM。
        3. RM 输出一个标量值（Logit），代表“人类喜欢这个回复的概率”。
    - 特点：较慢，需要额外的显存和计算资源，能捕捉复杂语义。
    
###### 第三步：分布式计算与聚合 (Distributed Computation)

在大规模训练中，奖励计算也是并行的。

- 动作：
    - 如果是 FSDP/Megatron 后端，Reward Worker 会在多个 GPU 上并行计算。
    - 如果是 Ray 后端，可能会启动多个 Ray Actor（Worker）并行处理不同的 Batch。
- 结果：每个 Worker 计算出一部分样本的分数，最后通过 all_gather 或 Ray 的 get() 聚合回主进程。

###### 第四步：Token 级奖励映射 (Token-Level Mapping) —— 这是关键点

这是新手最容易困惑的地方。

- 输入：模型生成的是一个序列（例如 100 个 Token）。
- 输出：打分函数通常只给整个序列打一个分（例如 1.0 分）。
- 问题：PPO 算法需要知道每个 Token 的奖励，才能更新策略。

compute_rewards 负责解决这个映射问题，通常采用 “稀疏奖励” 策略：
- 逻辑：将序列级别的分数，赋值给最后一个有效 Token（或 EOS token），其余 Token 的奖励设为 0（或 KL 惩罚值）。
- 代码逻辑示意： 意义：告诉模型，“虽然你说了 100 个词，但只有最后得出正确结论的那一刻，才是真正有价值的”。

In [ ]:
# 伪代码
# reward_tensor 初始化为 0
reward_tensor = torch.zeros_like(response_ids)

# 将计算出的标量分数 score 赋给最后一个位置
# valid_length 是去除 padding 后的实际长度
reward_tensor[batch_idx, valid_length - 1] = score

在 verl/workers/reward_worker.py 中，逻辑大致如下：

In [ ]:
def compute_rewards(self, prompts, responses):
    # 1. 拼接 Prompt 和 Response
    full_sequences = self.tokenizer.apply_chat_template(
        zip(prompts, responses), return_tensors="pt"
    ).to('cuda')
    
    # 2. 根据配置选择打分方式
    if self.config.use_reward_model:
        # --- 模型打分模式 ---
        # 调用 Reward Model 进行推理
        with torch.no_grad():
            outputs = self.rm_model(full_sequences)
            # 提取最后一个 token 的 logit 作为分数
            scores = outputs.logits[:, -1] 
    else:
        # --- 规则打分模式 ---
        # 遍历每个样本，用 Python 函数打分
        scores = []
        for prompt, response in zip(prompts, responses):
            score = self.rule_fn(prompt, response) # 例如 check_math_answer
            scores.append(score)
        scores = torch.tensor(scores)

    # 3. 处理分布式数据 (All Gather)
    # 如果是多卡，需要收集所有卡的分数
    global_scores = self.gather_scores(scores)
    
    return global_scores

##### 🌟 为什么它很重要？

compute_rewards 是 RLHF 的指挥棒。

- <font color='red'>决定方向</font>：<font color='green'>模型完全依赖这个分数来调整参数。如果奖励给错了（比如奖励了废话），模型就会学会说废话。</font>
- 性能瓶颈：在使用 Reward Model 时，这一步往往比生成还要慢（因为 RM 通常也是大模型）。<font color='red'>VeRL 通过流水线并行（生成完一个 Batch 立即计算奖励，而不是等所有都生成完）来优化这个问题。</font>

<font color='red'>总结来说，它是连接“模型输出”和“梯度更新”的桥梁，把抽象的“好/坏”变成了具体的数学信号。</font>

### 第四步：计算优势 (Compute Advantage)

这是 PPO 算法的数学核心。我们需要知道某个动作比“平均水平”好多少。
- 动作：Trainer 在本地（或调用算法工具类）执行。
- 算法：使用 GAE (Generalized Advantage Estimation) 算法。
    - 利用之前计算的 Values 和 Rewards，计算优势函数 At。
    - 公式大致为： At=δt+(γλ)δt+1+… ，其中 δ 是时序差分误差。
- 数据流向：Rewards + Values →→ GAE Algo →→ Advantages (A_t)。


#### 在 VeRL 框架中，计算优势 (Compute Advantage) 是连接“环境反馈”与“模型更新”的数学桥梁。
- 如果说 RewardWorker 是告诉模型“结果是对是错”，那么 优势计算 就是告诉模型“这个动作比平均水平好多少”。这一步通常发生在 trainer.fit() 的主循环中，位于奖励计算之后、模型更新之前。

###### 🎯 核心目标：解决“信用分配”问题

在强化学习中，模型生成了一整段话（序列），最后得到了一个分数（Reward）。
- 问题：这个分数是针对整段话的，但模型需要知道<font color='red'>具体是哪个词（Token） 导致了高分或低分。</font>
- 目标：计算出一个标量 At（优势），代表在时刻 t 采取的动作比“预期”好多少。


###### 💻 VeRL 中的实现逻辑

在 VeRL 的代码实现中（通常在 verl/trainer/ppo/core_trainer.py 或算法工具类中），这个过程是纯数学计算，不涉及梯度反向传播。

###### 第一步：准备数据
从之前的步骤中收集以下张量（通常已经由 CriticWorker 和 RewardWorker 计算好）：
- values: Critic 对每个 Token 的估值 V(s) 。
- rewards: 环境给出的奖励 r 。
- done_masks: 标记序列是否结束（用于处理截断）。

###### 第二步：逆向计算 (Backward Loop)

由于$ A_t依赖于 A_{t+1}$，<font color='red'>计算必须从序列的最后一个 Token 向前进行。</font>

In [ ]:
# 伪代码：VeRL 中的 GAE 计算逻辑
# values: [Batch, SeqLen]
# rewards: [Batch, SeqLen]
# gamma, lambda_gae: 超参数

last_gae = 0.0
advantages = torch.zeros_like(values)

# 从后往前遍历序列
for t in reversed(range(seq_len)):
    # 1. 计算 TD Error (delta)
    # next_value 如果是最后一个 token，则为 0
    next_value = values[:, t+1] if t < seq_len - 1 else 0.0
    
    delta = rewards[:, t] + gamma * next_value * (1 - done_masks[:, t]) - values[:, t]
    
    # 2. 累积 GAE
    # 公式：A_t = delta_t + gamma * lambda * A_{t+1}
    last_gae = delta + gamma * lambda_gae * (1 - done_masks[:, t]) * last_gae
    
    advantages[:, t] = last_gae

###### 第三步：计算回报 (Returns)
除了优势At，还需要计算总回报 Rt用于训练 Critic）。$ R_t=A_t+V(s_t) $


###### 🌟 为什么这步很关键？
1. 方差缩减：

如果不使用 GAE（即只用蒙特卡洛采样），梯度的波动会非常大，导致训练不稳定。GAE 引入了 Critic 的基线（Baseline），极大地平滑了训练曲线。
2. 稀疏奖励处理：

在数学推理（如 GSM8K）中，只有最后答案对了才有奖励（Sparse Reward）。GAE 能利用 Critic 的预测，将这个稀疏的奖励信号有效地传播回前面的推理步骤，告诉模型：“虽然这一步还没出结果，但这步推理是对的，因为它导向了最终的高分。”
3. 无梯度计算：

这一步在 VeRL 中是纯数值计算，不占用显存（不需要保存计算图），因此速度极快，通常不会成为瓶颈。

# fit函数

######  🧠 1. 核心流程概览
这段代码实现的是一个标准的 PPO 训练循环，流程如下：
1. 采样 (Rollout)：使用 Actor 策略生成文本（回答）。
2. 评估 (Reward)：计算生成文本的奖励值（结合 Reward Model 和规则）。
3. 计算 (Compute)：计算旧策略的 LogProb（对数概率）和 Critic 的价值（Values）。
4. 优化 (Update)：更新 Actor（策略网络）和 Critic（价值网络）。
5. 验证 (Validate)：定期在验证集上测试性能。

###### 🧩 2. 关键组件与角色
代码中通过 RPC 调用不同的“工作组”（Worker Group）来完成任务：

- actor_rollout_wg: 负责生成文本（generate_sequences）和计算对数概率（compute_log_prob）。
- rm_wg: Reward Model 工作组，负责打分。
- critic_wg: 负责计算状态价值（Values），用于优势估计。
- ref_policy_wg: 参考策略（通常是 SFT 模型或旧策略），用于计算 KL 散度惩罚。
- DataProto: 一个自定义的数据结构，用于在组件间高效传递张量（Tensors）和非张量数据。

###### 📖 3. 代码逻辑逐段详解

###### A. 初始化与检查点 (Initialization)
- 作用：恢复训练状态，并在正式训练前跑一次验证集作为基准。

In [ ]:
self._load_checkpoint() # 加载模型检查点
if self.config.trainer.get("val_before_train", True): 
    self._validate() # 训练前先验证一次

###### B. 数据加载与批处理 (Data Loading)
- 逻辑：从数据加载器读取数据。repeat 操作意味着对同一个 Prompt 生成 n 个不同的回答（Response），这是为了增加采样多样性。


In [ ]:
for epoch in range(...):
    for batch_dict in self.train_dataloader:
        # 处理 batch，分离出 prompt 用于生成
        gen_batch = new_batch.pop([...]) 
        gen_batch = gen_batch.repeat(repeat_times=n, interleave=True)

###### C. 生成阶段 (Rollout / Generation)
- 动作：Actor 模型根据 Prompt 生成完整的文本序列。

In [ ]:
gen_batch_output = self.actor_rollout_wg.generate_sequences(gen_batch)

###### D. 奖励计算 (Reward Computation)
这是代码中最复杂的部分，包含多种奖励机制的融合：

1. REM-Ax (可选)：如果使用 REM-Ax 算法，会生成一组贪婪解码（Greedy）的 Baseline 用于对比。
2. Reward Model 打分：

In [ ]:
reward_tensor = self.rm_wg.compute_rm_score(new_batch)

3. 规则奖励 (Rule-based)：

In [ ]:
reward_result = self.reward_fn(new_batch, return_dict=True)

4. KL 惩罚 (Crucial for PPO)：
- 融合：最终的奖励值 = RM 分数 + 规则分数 + KL 惩罚。

In [ ]:
apply_kl_penalty(...) # 计算 KL 散度惩罚，防止模型偏离原始 SFT 模型太远

###### E. 过滤机制 (Filtering)
- 逻辑：如果配置了过滤，它会根据生成结果的方差（Variance）或分数（Score）筛选样本。如果筛选后样本不够，它会跳过更新，回到循环继续生成（continue），直到凑够足够的高质量样本。

In [ ]:
if self.config.algorithm.filter_groups.enable:
    # 根据标准差或分数过滤掉不好的样本
    # 如果过滤后样本不足，会继续生成直到凑够 batch size

###### F. 优势估计与更新 (Advantage & Update)
当样本收集足够（Align the batch）后，进入更新阶段：

1. 计算旧概率：self.actor_rollout_wg.compute_log_prob(batch)（用于 PPO 公式中的旧策略概率）。
2. 计算参考对数概率（如果有）：用于 KL 计算。
3. 计算价值 (Values)：self.critic_wg.compute_values(batch)。
4. 计算优势 (Advantages)：

In [ ]:
compute_advantage(...) # 在 Driver 进程本地计算 (GAE)

5. 更新 Critic：self.critic_wg.update_critic(batch)。
6. 更新 Actor：self.actor_rollout_wg.update_actor(batch)。

###### G. 验证与保存 (Logging)

In [ ]:
if self.global_steps % self.config.trainer.test_freq == 0:
    self._validate() # 定期验证
self._save_checkpoint() # 定期保存
logger.log(data=metrics, step=self.global_steps) # 记录指标

###### ⚙️ 4. 特殊功能亮点

1. REM-Ax 支持：

代码中包含 if self.config.algorithm.adv_estimator == AdvantageEstimator.REMAX 的分支。REM-Ax 是一种改进的 PPO 变体，它利用贪婪解码的结果作为 Baseline 来减少方差。

2. 动态 Batch 过滤：

代码实现了“如果过滤后样本不足，则继续生成”的逻辑。这保证了训练时的 Batch Size 是动态的，但有效样本数是恒定的，这对于基于过滤的 RLHF 非常重要。

3. 分布式数据并行 (DP Balancing)：

代码中包含 _balance_batch 函数调用，这表明框架考虑了多 GPU 训练时的数据平衡问题。

4. Profiling (性能分析)：

使用了 marked_timer 装饰器来记录各个阶段（生成、奖励、更新）的耗时，便于性能调优。

In [ ]:
 def fit(self):
        """
        The training loop of PPO.
        The driver process only need to call the compute functions of the worker group through RPC
        to construct the PPO dataflow.
        The light-weight advantage computation is done on the driver process.
        """
        from omegaconf import OmegaConf

        from verl.utils.tracking import Tracking

        logger = Tracking(
            project_name=self.config.trainer.project_name,
            experiment_name=self.config.trainer.experiment_name,
            default_backend=self.config.trainer.logger,
            config=OmegaConf.to_container(self.config, resolve=True),
        )

        self.global_steps = 0
        self.gen_steps = 0

        # load checkpoint before doing anything
        self._load_checkpoint()

        # perform validation before training
        # currently, we only support validation using the reward_function.
        if self.val_reward_fn is not None and self.config.trainer.get("val_before_train", True):
            val_metrics = self._validate()
            assert val_metrics, f"{val_metrics=}"
            pprint(f"Initial validation metrics: {val_metrics}")
            logger.log(data=val_metrics, step=self.global_steps)
            if self.config.trainer.get("val_only", False):
                return

        if self.config.actor_rollout_ref.rollout.get("skip_rollout", False):
            rollout_skip = RolloutSkip(self.config, self.actor_rollout_wg)
            rollout_skip.wrap_generate_sequences()

        # add tqdm
        progress_bar = tqdm(total=self.total_training_steps, initial=self.global_steps, desc="Training Progress")

        # we start from step 1
        self.global_steps += 1
        self.gen_steps += 1
        last_val_metrics = None

        prev_step_profile = False
        curr_step_profile = (
            self.global_steps in self.config.trainer.profile_steps
            if self.config.trainer.profile_steps is not None
            else False
        )
        next_step_profile = False

        timing_raw = defaultdict(float)
        batch = None
        num_prompt_in_batch = 0
        num_gen_batches = 0
        for epoch in range(self.config.trainer.total_epochs):
            for batch_dict in self.train_dataloader:
                metrics = {}

                with marked_timer("start_profile", timing_raw):
                    self._start_profiling(
                        not prev_step_profile and curr_step_profile
                        if self.config.trainer.profile_continuous_steps
                        else curr_step_profile
                    )

                new_batch: DataProto = DataProto.from_single_dict(batch_dict)
                num_gen_batches += 1
                # pop those keys for generation
                if "multi_modal_data" in new_batch.non_tensor_batch.keys():
                    gen_batch = new_batch.pop(
                        batch_keys=["input_ids", "attention_mask", "position_ids"],
                        non_tensor_batch_keys=["raw_prompt_ids", "multi_modal_data"],
                    )
                else:
                    gen_batch = new_batch.pop(
                        batch_keys=["input_ids", "attention_mask", "position_ids"],
                        non_tensor_batch_keys=["raw_prompt_ids"],
                    )
                gen_batch = gen_batch.repeat(repeat_times=self.config.actor_rollout_ref.rollout.n, interleave=True)

                is_last_step = self.gen_steps >= self.total_training_steps

                with marked_timer("step", timing_raw):
                    # generate a batch
                    with marked_timer("gen", timing_raw, "red"):
                        gen_batch_output = self.actor_rollout_wg.generate_sequences(gen_batch)
                        timing_raw.update(gen_batch_output.meta_info["timing"])
                        gen_batch_output.meta_info.pop("timing", None)

                    if self.config.algorithm.adv_estimator == AdvantageEstimator.REMAX:
                        with marked_timer("gen_max", timing_raw, "red"):
                            gen_baseline_batch = deepcopy(gen_batch)
                            gen_baseline_batch.meta_info["do_sample"] = False
                            gen_baseline_output = self.actor_rollout_wg.generate_sequences(gen_baseline_batch)

                            new_batch = new_batch.union(gen_baseline_output)
                            reward_baseline_tensor = self.reward_fn(new_batch)
                            reward_baseline_tensor = reward_baseline_tensor.sum(dim=-1)

                            new_batch.pop(batch_keys=list(gen_baseline_output.batch.keys()))

                            new_batch.batch["reward_baselines"] = reward_baseline_tensor

                            del gen_baseline_batch, gen_baseline_output

                    new_batch.non_tensor_batch["uid"] = np.array(
                        [str(uuid.uuid4()) for _ in range(len(new_batch.batch))], dtype=object
                    )
                    # repeat to align with repeated responses in rollout
                    new_batch = new_batch.repeat(repeat_times=self.config.actor_rollout_ref.rollout.n, interleave=True)
                    new_batch = new_batch.union(gen_batch_output)

                    with marked_timer("reward", timing_raw, "yellow"):
                        # compute scores. Support both model and function-based.
                        # We first compute the scores using reward model. Then, we call reward_fn to combine
                        # the results from reward model and rule-based results.
                        if self.use_rm:
                            # we first compute reward model score
                            reward_tensor = self.rm_wg.compute_rm_score(new_batch)
                            new_batch = new_batch.union(reward_tensor)

                        # we combine with rule-based rm
                        reward_extra_infos_dict: dict[str, list]
                        try:
                            reward_result = self.reward_fn(new_batch, return_dict=True)
                            reward_tensor = reward_result["reward_tensor"]
                            reward_extra_infos_dict = reward_result.get("reward_extra_info", {})
                        except Exception as e:
                            print(f"Error in reward_fn: {e}")
                            reward_tensor = self.reward_fn(new_batch)
                            reward_extra_infos_dict = {}

                        new_batch.batch["token_level_scores"] = reward_tensor
                        
                        # check 2k filtered acc
                        if reward_extra_infos_dict:
                            new_batch.non_tensor_batch.update(
                                {k: np.array(v) for k, v in reward_extra_infos_dict.items()}
                            )

                        # compute rewards. apply_kl_penalty if available
                        if self.config.algorithm.use_kl_in_reward:
                            new_batch, kl_metrics = apply_kl_penalty(
                                new_batch, kl_ctrl=self.kl_ctrl_in_reward, kl_penalty=self.config.algorithm.kl_penalty
                            )
                            metrics.update(
                                kl_metrics
                            )  # TODO: This will be cleared if we use multiple genenration batches
                        else:
                            new_batch.batch["token_level_rewards"] = new_batch.batch["token_level_scores"]

                    if not self.config.algorithm.filter_groups.enable:
                        batch = new_batch
                    else:  # NOTE: When prompts after filtering is less than train batch size,
                        # we skip to the next generation batch
                        metric_name = self.config.algorithm.filter_groups.metric
                        if metric_name == "seq_final_reward":
                            # Turn to numpy for easier filtering
                            new_batch.non_tensor_batch["seq_final_reward"] = (
                                new_batch.batch["token_level_rewards"].sum(dim=-1).numpy()
                            )
                        elif metric_name == "seq_reward":
                            new_batch.non_tensor_batch["seq_reward"] = (
                                new_batch.batch["token_level_scores"].sum(dim=-1).numpy()
                            )

                        # Collect the sequence reward for each trajectory
                        prompt_uid2metric_vals = defaultdict(list)
                        for uid, metric_val in zip(
                            new_batch.non_tensor_batch["uid"], new_batch.non_tensor_batch[metric_name], strict=True
                        ):
                            prompt_uid2metric_vals[uid].append(metric_val)

                        prompt_uid2metric_std = {}
                        for prompt_uid, metric_vals in prompt_uid2metric_vals.items():
                            prompt_uid2metric_std[prompt_uid] = np.std(metric_vals)

                        kept_prompt_uids = [
                            uid
                            for uid, std in prompt_uid2metric_std.items()
                            if std > 0 or len(prompt_uid2metric_vals[uid]) == 1
                        ]
                        num_prompt_in_batch += len(kept_prompt_uids)

                        kept_traj_idxs = []
                        for idx, traj_from_prompt_uid in enumerate(new_batch.non_tensor_batch["uid"]):
                            if traj_from_prompt_uid in kept_prompt_uids:
                                kept_traj_idxs.append(idx)

                        new_batch = new_batch[kept_traj_idxs]
                        batch = new_batch if batch is None else DataProto.concat([batch, new_batch])

                        prompt_bsz = self.config.data.train_batch_size
                        if num_prompt_in_batch < prompt_bsz:
                            print(f"{num_prompt_in_batch=} < {prompt_bsz=}")
                            max_num_gen_batches = self.config.algorithm.filter_groups.max_num_gen_batches
                            if max_num_gen_batches <= 0 or num_gen_batches < max_num_gen_batches:
                                print(f"{num_gen_batches=}. Keep generating...")
                                progress_bar.update(1)
                                self.gen_steps += 1
                                continue
                            else:
                                raise ValueError(
                                    f"{num_gen_batches=} >= {max_num_gen_batches=}."
                                    + " Generated too many. Please check if your data are too difficult."
                                    + " You could also try set max_num_gen_batches=0 to enable endless trials."
                                )
                        else:
                            # Align the batch
                            traj_bsz = self.config.data.train_batch_size * self.config.actor_rollout_ref.rollout.n
                            batch = batch[:traj_bsz]

                    # === Updating ===

                    batch.batch["response_mask"] = compute_response_mask(batch)

                    # Balance the number of valid tokens across DP ranks.
                    # NOTE: This usually changes the order of data in the `batch`,
                    # which won't affect the advantage calculation (since it's based on uid),
                    # but might affect the loss calculation (due to the change of mini-batching).
                    # TODO: Decouple the DP balancing and mini-batching.
                    if self.config.trainer.balance_batch:
                        self._balance_batch(batch, metrics=metrics)

                    # compute global_valid tokens
                    batch.meta_info["global_token_num"] = torch.sum(batch.batch["attention_mask"], dim=-1).tolist()

                    # recompute old_log_probs
                    with marked_timer("old_log_prob", timing_raw, "blue"):
                        old_log_prob = self.actor_rollout_wg.compute_log_prob(batch)
                        entropys = old_log_prob.batch["entropys"]
                        response_masks = batch.batch["response_mask"]
                        loss_agg_mode = self.config.actor_rollout_ref.actor.loss_agg_mode
                        entropy_agg = agg_loss(loss_mat=entropys, loss_mask=response_masks, loss_agg_mode=loss_agg_mode)
                        old_log_prob_metrics = {"actor/entropy": entropy_agg.detach().item()}
                        metrics.update(old_log_prob_metrics)
                        old_log_prob.batch.pop("entropys")
                        batch = batch.union(old_log_prob)

                    if self.use_reference_policy:
                        # compute reference log_prob
                        with marked_timer("ref", timing_raw, "olive"):
                            ref_log_prob = self.ref_policy_wg.compute_ref_log_prob(batch)
                            batch = batch.union(ref_log_prob)

                    # compute values
                    if self.use_critic:
                        with marked_timer("values", timing_raw, "cyan"):
                            values = self.critic_wg.compute_values(batch)
                            batch = batch.union(values)

                    with marked_timer("adv", timing_raw, "brown"):
                        # compute advantages, executed on the driver process
                        norm_adv_by_std_in_grpo = self.config.algorithm.get("norm_adv_by_std_in_grpo", True)
                        batch = compute_advantage(
                            batch,
                            adv_estimator=self.config.algorithm.adv_estimator,
                            gamma=self.config.algorithm.gamma,
                            lam=self.config.algorithm.lam,
                            num_repeat=self.config.actor_rollout_ref.rollout.n,
                            norm_adv_by_std_in_grpo=norm_adv_by_std_in_grpo,
                        )

                    # update critic
                    if self.use_critic:
                        with marked_timer("update_critic", timing_raw, "pink"):
                            critic_output = self.critic_wg.update_critic(batch)
                        critic_output_metrics = reduce_metrics(critic_output.meta_info["metrics"])
                        metrics.update(critic_output_metrics)

                    # implement critic warmup
                    global_log_buffer = None
                    # 默认值为0
                    if self.config.trainer.critic_warmup <= self.global_steps:
                        # update actor
                        with marked_timer("update_actor", timing_raw, "red"):
                            actor_output = self.actor_rollout_wg.update_actor(batch)
                        
                        # Handle global logging for negative_approx_kl
                        if "global_log_buffer" in actor_output.non_tensor_batch:
                            global_log_buffer = actor_output.non_tensor_batch.pop("global_log_buffer")

                        actor_output_metrics = reduce_metrics(actor_output.meta_info["metrics"])
                        metrics.update(actor_output_metrics)

                    # NOTE: We copy the rollout saving from ray_ppo_trainer.py#L1282.
                    # Log rollout generations if enabled
                    rollout_data_dir = self.config.trainer.get("rollout_data_dir", None)
                    if rollout_data_dir:
                        with marked_timer("dump_rollout_generations", timing_raw, color="green"):
                            inputs = self.tokenizer.batch_decode(batch.batch["prompts"], skip_special_tokens=True)
                            outputs = self.tokenizer.batch_decode(batch.batch["responses"], skip_special_tokens=True)
                            scores = batch.batch["token_level_scores"].sum(-1).cpu().tolist()
                            sample_gts = [
                                item.non_tensor_batch.get("reward_model", {}).get("ground_truth", None)
                                for item in batch
                            ]

                            if "request_id" in batch.non_tensor_batch:
                                reward_extra_infos_dict.setdefault(
                                    "request_id",
                                    batch.non_tensor_batch["request_id"].tolist(),
                                )

                            self._dump_generations(
                                inputs=inputs,
                                outputs=outputs,
                                gts=sample_gts,
                                scores=scores,
                                reward_extra_infos_dict=reward_extra_infos_dict,
                                dump_path=rollout_data_dir,
                            )
                        
                        # Save negative_approx_kl to the same directory
                        if global_log_buffer is not None:
                            import json
                            import os
                            save_path = os.path.join(rollout_data_dir, f"negative_approx_kl_global_step_{self.global_steps}.jsonl")
                            
                            with open(save_path, "w") as f:
                                for item in global_log_buffer:
                                    # item is { "negative_approx_kl": tensor, "responses": tensor }
                                    neg_kl = item["negative_approx_kl"]
                                    resps = item["responses"]
                                    
                                    if isinstance(neg_kl, torch.Tensor):
                                        neg_kl = neg_kl.tolist()
                                    if isinstance(resps, torch.Tensor):
                                        resps = resps.tolist()
                                    
                                    # Try to flatten if possible, or just save as batch
                                    # If neg_kl is a list of values and resps is list of responses
                                    if isinstance(neg_kl, list) and isinstance(resps, list) and len(neg_kl) == len(resps) and not isinstance(neg_kl[0], list):
                                         for nk, r in zip(neg_kl, resps):
                                             json.dump({"negative_approx_kl": nk, "response": r}, f)
                                             f.write("\n")
                                    else:
                                        json.dump({"negative_approx_kl": neg_kl, "responses": resps}, f)
                                        f.write("\n")

                    # validate
                    if (
                        self.val_reward_fn is not None
                        and self.config.trainer.test_freq > 0
                        and (is_last_step or self.global_steps % self.config.trainer.test_freq == 0)
                    ):
                        with marked_timer("testing", timing_raw, "green"):
                            val_metrics: dict = self._validate()
                            if is_last_step:
                                last_val_metrics = val_metrics
                        metrics.update(val_metrics)

                    if self.config.trainer.save_freq > 0 and (
                        is_last_step or self.global_steps % self.config.trainer.save_freq == 0
                    ):
                        with marked_timer("save_checkpoint", timing_raw, "green"):
                            self._save_checkpoint()

                with marked_timer("stop_profile", timing_raw):
                    next_step_profile = (
                        self.global_steps + 1 in self.config.trainer.profile_steps
                        if self.config.trainer.profile_steps is not None
                        else False
                    )
                    self._stop_profiling(
                        curr_step_profile and not next_step_profile
                        if self.config.trainer.profile_continuous_steps
                        else curr_step_profile
                    )
                    prev_step_profile = curr_step_profile
                    curr_step_profile = next_step_profile

                # collect metrics
                metrics.update(compute_data_metrics(batch=batch, use_critic=self.use_critic))
                metrics.update(compute_timing_metrics(batch=batch, timing_raw=timing_raw))
                # TODO: implement actual tflpo and theoretical tflpo
                n_gpus = self.resource_pool_manager.get_n_gpus()
                metrics.update(compute_throughout_metrics(batch=batch, timing_raw=timing_raw, n_gpus=n_gpus))
                timing_raw = defaultdict(float)  # clear timing

                metrics["train/num_gen_batches"] = num_gen_batches
                batch = None
                num_prompt_in_batch = 0
                num_gen_batches = 0

                # TODO: make a canonical logger that supports various backend
                logger.log(data=metrics, step=self.global_steps)

                if is_last_step:
                    pprint(f"Final validation metrics: {last_val_metrics}")
                    progress_bar.close()
                    return

                progress_bar.update(1)
                self.global_steps += 1
                self.gen_steps += 1

### 第五步：策略与价值更新 (Update Policy & Value)

拿到了“优势”这个指导信号，终于可以更新模型参数了。
- 动作：
    - 调用：ActorWorker.update_policy(prompts,responses,rewards,old_log_probs)
    - 更新 Critic：Trainer 指示 Critic Worker 进行梯度下降，让它预测的价值 V(s) 更接近真实的回报。
    - 更新 Actor：Trainer 指示 Actor Worker 进行梯度上升，最大化优势函数。
        - 这里会执行 PPO 的截断机制（Clipping），防止策略更新幅度过大导致训练崩塌。
        - 同时会加入 KL 惩罚项（参考 Reference Worker 的结果）。
- 数据流向：Advantages →→ Critic/Actor →→ Updated Weights。
- 核心算法（FIPO）:
    - loss计算： 进入policy_loss模块
    - future KL逻辑
        - 根据loss_mode=future_kl,计算当前token与未来token的KL散度
        - 应用decay_rate=32.0 和chunk_size = 128
    - GRPO逻辑： 计算组内优势，即比较同一个prompt下16个回复的奖励差异
    - 反向传播：计算梯度，更新actor模型参数


####  ActorWorker.update_policy(prompts, responses, rewards, old_log_probs) 是 VeRL 训练循环中的“学习时刻”。

<font color='red'>如果说前面的步骤（生成、打分、计算优势）都是在“收集经验”，那么 update_policy 就是“反思与进化”。这是整个强化学习流程中，唯一真正修改 Actor 模型权重的步骤。</font>

在这个过程中，Actor 利用计算好的优势（Advantages），通过 PPO（或 GRPO）算法，告诉模型：“下次遇到类似的情况，要多生成那些能拿高分的词，少生成那些导致低分的词。”

以下是 ActorWorker.update_policy() 的详细执行逻辑拆解：


###### 🎯 核心功能：它做什么？

简单来说，它接收一批“经验数据”（Prompts, Responses）和“指导信号”（Rewards, Advantages），执行以下步骤：

1. 计算损失 (Loss)：使用 PPO/GRPO 公式计算当前策略与旧策略的差异。
2. 反向传播 (Backward)：计算梯度。
3. 参数更新 (Step)：利用优化器（如 AdamW）更新模型权重。

###### ⚙️ 执行流程：四步走
当 trainer 调用 actor_worker.update_policy(batch) 时，底层会发生以下关键步骤：

###### 第一步：数据准备与预处理 (Data Prep)

这一步主要负责将数据搬运到 GPU，并处理分布式并行策略（如 FSDP 或 Megatron）。

- 动作：
    - 数据上云：将 prompts, responses, rewards, advantages, old_log_probs 等张量移动到 CUDA 设备上。
    - 并行切分：如果开启了序列并行（Sequence Parallelism, 如 Ulysses），数据会在序列维度上被切分，分发到不同的 GPU 上。
    - 卸载处理：如果开启了参数卸载（Offload），此时会先将 FSDP 包装的模型参数和优化器状态从 CPU 加载回 GPU。
    
###### 第二步：前向传播与损失计算 (Forward & Loss)
这是算法的核心。Actor 需要重新“看”一遍生成的回复，计算当前的概率，并与“旧概率”做对比。
逻辑：
1. <font color='red'>重算 Log Probs：</font>
    - 将 prompts + responses 输入到当前的 Actor 模型中。
    - 输出当前的对数概率 logπ_θ(a_t∣s_t) 。
    - <font color='red'>注意：这里必须重新计算，因为模型参数在上一次更新后已经变了。</font>
2. 计算比率 (Ratio)：
    - 计算当前概率与 old_log_probs（生成数据时的概率）的比值
    
    
###### 第三步：反向传播 (Backward)
- 动作：
    - 调用 loss.backward()。
    - 计算模型参数相对于损失的梯度。
    - <font color='red'>在 FSDP 模式下，这一步会自动触发跨卡的梯度同步（All-Reduce），确保所有 GPU 上的梯度是一致的。</font>
    
###### 第四步：优化器更新 (Optimizer Step)
这是真正的“参数修改”时刻。

- 动作：
    - 调用 optimizer.step()。
    - 利用计算好的梯度，更新模型权重θ 。
    - 更新学习率调度器（Learning Rate Scheduler）。
    - 清零梯度：调用 optimizer.zero_grad()，为下一次迭代做准备。
    
###### 第五步：资源回收与卸载 (Cleanup & Offload)
- 动作：
    - 如果开启了参数卸载，更新完成后，会将模型参数和优化器状态重新卸载回 CPU，释放 GPU 显存，为下一轮的 Rollout 生成腾出空间。
    - 清空 CUDA 缓存 (torch.cuda.empty_cache())。
    
在 verl/workers/actor/dp_actor.py 中，逻辑大致如下：

In [ ]:
def update_policy(self, data):
    # 1. 数据预处理 (切分、上云)
    data = self.ulysses_sharding_manager.preprocess_data(data)
    
    # 2. 开启梯度计算
    self.optimizer.zero_grad()
    
    # 3. 循环更新 (PPO Epochs)
    # 通常在一个 Batch 上会重复训练多次 (ppo_epochs)
    for _ in range(self.config.ppo_epochs):
        
        # A. 前向传播：计算当前的 log_probs
        # 输入: prompts + responses
        # 输出: 新的 log_probs
        current_log_probs = self.model(data.input_ids).log_probs
        
        # B. 计算损失
        # 使用 PPO 或 GRPO 的特定损失函数
        # 输入: current_log_probs, old_log_probs, advantages
        loss = self.compute_ppo_loss(
            current_log_probs, 
            data.old_log_probs, 
            data.advantages
        )
        
        # C. 反向传播
        loss.backward()
        
    # 4. 更新参数
    self.optimizer.step()
    self.lr_scheduler.step()
    
    # 5. 后处理 (卸载、切分恢复)
    self.ulysses_sharding_manager.postprocess_data()
    
    return loss.item()

###### 🌟 为什么它很重要？
update_policy 是 VeRL 的“进化引擎”。

- 策略迭代：<font color='red'>它通过“重要性采样”（Importance Sampling），利用旧数据（old_log_probs）来优化新策略，使得模型可以在不重新生成数据的情况下进行多次梯度下降，极大地提高了样本效率。</font>
- 稳定性：通过 PPO 的截断机制（Clipping），它限制了策略更新的幅度，防止模型因为一次巨大的梯度更新而“崩溃”（Mode Collapse）。
- <font color='red'>分布式协同：在 FSDP/Megatron 后端下，这一步协调了成百上千张 GPU 的梯度同步，确保大家往同一个方向“进化”。</font>

总结来说，这是整个 RLHF 循环的终点，也是新循环的起点——模型在这里变得更聪明了一点。

# 第四阶段：结束与保存

## 循环检查与保存
    - 每trainer.save_freq=5步，调用checkpoint_manager.save_checkpoint()
    - 将模型权重保存到CKPTS_DIR
    
## 程序退出
    - 训练轮次结束
    - ray任务完成，进程退出
    
###### 在 VeRL 的训练循环中，每 trainer.save_freq=5步，调用 checkpoint_manager.save_checkpoint() 这一机制是整个训练任务的“时光机”和“保险箱”。

<font color='red'>考虑到大规模 RL 训练通常耗时数天甚至数周，且占用昂贵的 GPU 资源，这一步至关重要。它确保了即使发生硬件故障或断电，训练也能从最近的进度恢复，而不需要从头再来。</font>

###### 🎯 核心目标：不仅仅是保存模型

###### checkpoint_manager.save_checkpoint() 的任务远比简单的 torch.save(model.state_dict()) 复杂。<font color='red'>它需要保存的是整个训练系统的完整状态，以便实现“断点续训”。</font>

它主要保存以下四个维度的数据：

1. <b>模型权重</b>：Actor 和 Critic 的参数。
2. <b>优化器状态</b>：AdamW 的动量（momentum）和方差（variance）等，这对收敛至关重要。
3. <b>训练元数据</b>：当前的步数（global_step）、学习率调度器的状态、随机数种子（RNG seed）、数据加载器的偏移量（确保数据不重复、不遗漏）。
4. <b>Rollout 缓存（可选）</b>：在某些配置下，还会保存生成的经验数据，防止数据丢失。


###### ⚙️ 执行流程：<font color='red'>分层异步快照</font>

VeRL 采用了“分层异步快照”策略，旨在最大程度减少对训练速度的影响。<font color='red'>当 global_steps % save_freq == 0 时，触发以下流程：</font>

###### 第一步：触发检查 (Trigger Check)
在 trainer.fit() 的主循环中，每完成一个 PPO 更新步（Step），都会进行判断：

In [ ]:
# 伪代码：trainer.fit() 循环中
if (global_steps + 1) % self.config.trainer.save_freq == 0:
    # 触发保存
    self.checkpoint_manager.save_checkpoint(global_steps)

###### 第二步：状态收集与分片 (State Collection & Sharding)
<font color='red'>在分布式环境（FSDP/Megatron）下，模型参数是分散在各个 GPU 上的。</font>
- 动作：
    - CheckpointManager 会协调各个 Worker（Actor/Critic）。
    - <font color='red'>利用 FSDP 的 shard_state_dict 特性，只保存当前 GPU 负责的那一部分参数，而不是把所有参数收集到主进程再保存。</font>
    - 优势：这极大地降低了显存峰值压力和通信开销，避免了“保存模型时显存爆炸”的问题。

###### 第三步：异步写入 (Asynchronous Writing)
这是 VeRL 优化的关键点。
- 动作：
    - 保存操作被放入一个后台线程或独立进程中执行。
    - <font color='red'>主训练进程（Main Process）在触发保存后，不会被阻塞，而是继续执行下一个 Step 的计算。</font>
    - 后台进程负责将数据写入磁盘（本地 SSD 或远程 HDFS/S3）。
- 意义：即使保存一个 70B 的模型需要几分钟，也不会拖慢训练的迭代速度。

###### 第四步：元数据与软链接管理 (Meta & Symlink)
- 动作：
    - 保存一个 meta.json 文件，记录当前是第几步、配置是什么。
    - 更新目录下的 latest 软链接，指向最新的检查点文件夹（例如 checkpoint-1000）。
    - 清理旧文件：如果配置了 save_total_limit，管理器会自动删除最旧的检查点，防止磁盘爆满。
    
    
###### 📂 目录结构示例

执行保存后，磁盘上的目录结构通常如下所示：

In [ ]:
/checkpoints/
└── llama3-ppo-exp/
    ├── checkpoint-5/          # 第 5 步的检查点
    │   ├── actor/             # Actor 模型分片
    │   │   ├── fsdp_rank_0.pt
    │   │   └── ...
    │   ├── critic/            # Critic 模型分片
    │   ├── optimizer/         # 优化器状态
    │   └── meta.json          # 元数据
    ├── checkpoint-10/         # 第 10 步的检查点
    └── latest -> checkpoint-10/  # 指向最新的软链接

###### 🛡️ 容错与恢复 (Fault Tolerance)

这个机制是 VeRL 容错体系的核心。

- 自动恢复：<font color='red'>当训练意外中断（如节点宕机）后，重新启动脚本时，CheckpointManager 会检测 latest 链接。</font>
- 一致性校验：它会读取 meta.json，校验步数和配置是否匹配。
- 状态还原：
    - 加载模型权重。
    - 关键点：加载优化器状态，确保动量不丢失。
    - 关键点：恢复 RNG 种子和数据集偏移量，确保数据采样的随机性与之前保持一致，避免数据倾斜。

###### 📌 总结

<font color='red'>每 5 步保存一次 不仅仅是一个定时任务，它是 VeRL 工业级稳定性的体现。</font>

- 频率策略：save_freq=5 是一个平衡点。太频繁会占用 I/O 带宽，太稀疏会导致故障时回退太远。
- <font color='red'>核心价值：通过分片保存和异步写入，它实现了“无感备份”，让千卡规模的训练任务能够像“九命猫”一样，即使遇到故障也能迅速复活继续训练。</font>



## 🛠️ 辅助功能：不仅仅是训练

除了上述核心循环，trainer.fit() 还负责处理训练过程中的“杂务”：

- 日志记录 (Logging)：

每隔一定步数（Steps），它会收集各个 Worker 汇报的指标（Loss, Reward 均值, KL 散度等），并通过 Wandb 或 TensorBoard 展示出来。

- 检查点管理 (Checkpointing)：

它会定期检查当前进度。如果满足保存条件，它会触发 save_checkpoint()，将 Actor 和 Critic 的权重持久化到磁盘，以便断点续训。

- 验证 (Validation)：

在训练间隙，它可能会暂停训练，在验证集上跑一轮 validate()，看看模型在未见过的数据上表现如何，防止过拟合


## 📌 总结

trainer.fit() 是 VeRL 的总导演。它不直接做数学计算（那是 Worker 的事），也不直接管理 GPU 进程（那是 Ray 的事），它做的是编排：

1. 拿数据（Prompts）
2. 造数据（Rollout）
3. 评数据（Critic/Reward）
4. 算梯度（Advantage）
5. 改模型（Update）

正是这个函数的不断循环，让大模型从“胡说八道”逐渐变得“言之有物”。